# 9장. Skills로 분석 절차와 코드 재사용하기

기본 CSV 분석 절차와 실행 코드를 Skill로 재사용합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | Skill ZIP·등록과 실제 번들 스크립트 실행 |
| 이번 실습 조합 | CSV 분석 Skill + 기본 보고서 + skill_evidence.json |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

| 구성 | 재사용하는 것 | 이 교재의 예 |
| --- | --- | --- |
| Function Tool·MCP | 호출 가능한 업무 기능 | 목표 매출 조회 |
| Skill | 작업 절차·참고 자료·실행 코드 | CSV 분석·보고서 생성 |

Skill은 다른 도구와 함께 쓸 수 있는 작업 방법의 묶음입니다. 이번 장은 `csv-report`를 명시적으로 지정하고 번들 스크립트를 실행하도록 요청합니다. 자연어 요청만으로 Skill을 자동 선택하는 능력을 검증하는 실습은 아닙니다.

등록된 Skill과 실제 사용은 구분합니다. 실행 후 등록 metadata, 같은 Turn의 명령 이력, 내려받은 산출물을 각각 확인합니다. 기본 이미지의 전체 Skill 목록과도 구분합니다.

수동 실습은 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_extensions.py --chapter 9 --live`로 실행합니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 환경 준비와 실행 모드

로컬에서는 `uv sync --locked` 후 `.venv` 커널을 선택합니다. Colab에서는 다운로드 안내의 독립 실행본을 업로드합니다. 준비 셀은 패키지를 직접 설치하고 포함 파일을 생성합니다.
아래 RUN_LIVE를 True로 바꿨을 때만 실제 API 호출·sandbox 생성이 발생합니다.
기존 `.env.local`의 키·모델을 재사용하며 키는 sandbox·ZIP·로그에 넣지 않습니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
CHAPTER = "09_skills"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print({"live_enabled": settings.enabled, "model": settings.model or "Offline mode"})


## 1. Skill 구성과 입력 확인하기

```text
csv-report/
├── SKILL.md
├── references/
│   └── report-schema.md
└── scripts/
    └── analyze_sales.py
```

| 파일 | 역할 |
| --- | --- |
| `SKILL.md` | 언제 사용하고 어떤 절차를 따를지 안내 |
| `references/report-schema.md` | 입력·결과 파일의 계약 |
| `scripts/analyze_sales.py` | 결정적인 집계와 세 파일 생성 |

다음 셀은 입력·manifest·결과 계약·스크립트를 읽어 표시합니다. 로컬에서 분석 스크립트를 실행하거나 API를 호출하지 않습니다. 스크립트의 입력 읽기, 집계, JSON/보고서/증거 파일 생성 부분을 확인하세요.


In [ ]:
input_path = settings.root / "data/sales.csv"
csv_bytes = input_path.read_bytes()
csv_text = csv_bytes.decode("utf-8")
folder = settings.root / "skills/csv-report"
print(csv_text)
for name in ("SKILL.md", "references/report-schema.md", "scripts/analyze_sales.py"):
    print(f"\n--- {name} ---")
    print((folder / name).read_text(encoding="utf-8"))


## 2. ZIP과 inline Skill 요청 구성하기

먼저 Skill 이름·설명을 읽고 ZIP을 만듭니다. `read_skill_metadata()`는 이 예제의 인용 부호 없는 한 줄짜리 name/description만 읽는 boilerplate helper이며 일반 YAML 파서가 아닙니다. 복잡한 front matter는 명확한 오류로 거절합니다. API 요청 스키마는 다음 셀에 직접 표시합니다.

ZIP은 최상위 `csv-report/` 하나와 그 안의 manifest·참고 자료·코드를 포함합니다. 전체 실행 환경이나 가상환경을 넣지 않습니다. `archive_folder()`는 파일 포장만 수행합니다. ZIP 목록·바이트 수·SHA-256은 로컬 번들을 식별하는 정보이며 원격 실행 성공의 증거가 아닙니다.


In [ ]:
from tutorial_helpers.extensions import archive_folder, read_skill_metadata
import base64
import hashlib
import io
import zipfile

metadata = read_skill_metadata(folder)
raw = archive_folder(folder)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    print("ZIP entries:", archive.namelist())
print({"name": metadata["name"], "description": metadata["description"],
       "zip_bytes": len(raw), "zip_sha256": hashlib.sha256(raw).hexdigest()})


### ZIP 바이트와 API 필드 연결하기

| 필드 | 내용 |
| --- | --- |
| `name`, `description` | manifest에서 읽은 이름·설명 |
| `source.type` | 일반 base64 형식 |
| `source.media_type` | ZIP 형식 |
| `source.data` | 인코딩한 ZIP 바이트 |
| `environment.skills` | 이번 환경에 제공할 Skill 목록 |

이 장은 세션 요청에 inline ZIP을 첨부합니다. 별도 Skills API 업로드 개체를 만들거나 저장된 Skill 버전을 선택하지 않습니다. 공식 환경 문서는 skills·plugins·capability directories를 함께 소개하므로 이번 요청 방식과 구분해 읽으세요. [환경 가이드](https://developers.openai.com/api/docs/guides/agents-api/environments/openai-hosted#add-skills-and-plugins)


In [ ]:
skill = {
    "type": "inline",
    "name": metadata["name"],
    "description": metadata["description"],
    "source": {
        "type": "base64",
        "media_type": "application/zip",
        "data": base64.b64encode(raw).decode("ascii"),
    },
}
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
    "files": [{"type": "inline", "path": "/workspace/sales.csv",
               "data": base64.b64encode(csv_bytes).decode("ascii")}],
    "skills": [skill],
}
print({"skill_name": skill["name"], "skill_type": skill["type"],
       "source_type": skill["source"]["type"], "media_type": skill["source"]["media_type"],
       "sandbox_input": environment["files"][0]["path"]})


### 실행 요청과 의존성 구분하기

이 Skill은 표준 라이브러리만 사용하므로 원격 패키지 설치가 필요 없습니다. 외부 패키지가 필요한 Skill은 6장의 환경 준비처럼 의존성을 따로 준비합니다. ZIP에 가상환경을 넣는 방식과 구분합니다.

설치된 Skill의 절대 경로는 하드코딩하지 않습니다. 에이전트에게 Skill 지침을 읽고 그 디렉터리에서 번들 스크립트를 찾아 명시적인 `--input`·`--output-dir` 인자로 실행하도록 요청합니다. 분석 코드를 새로 작성하지 않도록 지시하고 `METRICS_CONTRACT`를 실제 입력에 조합합니다.


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)

task = (
    "Use the csv-report skill. Read its SKILL.md and locate its bundled "
    "scripts/analyze_sales.py relative to the skill directory. Run that script with Python "
    "using --input /workspace/sales.csv --output-dir /workspace/outputs. "
    "Do not rewrite the analyzer. Read summary.json, report.md, and skill_evidence.json back. "
    + METRICS_CONTRACT
)


## 3. Skill을 첨부해 분석 실행하기

준비한 환경을 `sessions.create()`에 전달해 새 분석을 시작합니다. inline Skill 첨부·환경 준비·스크립트 실행은 원격 단계이며, 앞의 준비 셀에서 로컬 분석을 실행한 것이 아닙니다.

각 실행·결과 단계는 오류·일반적인 인터럽트에서 `cleanup_on_error()`로 정리를 시도합니다. 정상 경로에서는 **8절 정리 셀까지 실행**하고 중간에 학습을 멈출 때도 정리 셀을 실행하세요. 강제 커널 종료는 자동 정리를 보장하지 못합니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

복구 기록은 `outputs/extensions/<chapter>/<run_id>/`, 다운로드는 그 아래 `downloads/<turn_id>/`에 저장합니다. 새 실행 시작 시 상태 준비를 한 번만 실행하세요. 진행 중 재실행하면 자원 식별자를 잃을 수 있습니다.

타이머는 실행 직전 시작해 셀 사이 학습 시간을 제외합니다. 600초 제한은 이벤트 도착 시 확인하므로 무응답 중 강제 중단하는 전체 타이머가 아닙니다. HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import json
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/extensions" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = None
downloads = None
downloads_complete = False
installed_environment = None


### 세션 생성과 응답 스트리밍

텍스트 delta를 표시하며 최종 응답을 수집하고, 스트림을 닫은 뒤 root Turn을 조회합니다. 새 작업의 중복 제출은 차단합니다. 실패·취소를 감지하고 예상하지 않은 외부 action은 오류로 처리합니다.

Turn 완료·모델의 Skill 언급·JSON 파싱만으로 Skill 사용이 확인됐다고 판단하지 않습니다. 다음 두 절에서 등록과 실제 명령을 따로 확인합니다. 이벤트 수·순서를 고정하거나 `idle`만으로 성공을 판단하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment=environment,
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})

    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15
    )
    if result.turn.status != "completed" or turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nCompleted analysis:")
    print({"session_id": result.session_id, "turn_id": result.turn_id, "status": turn.status})
    print(json.dumps(json.loads(result.output_text), indent=2))


## 4. 이번 환경의 Custom Skill 등록 확인하기

생성 이벤트의 Environment ID로 `environments.retrieve()`를 호출합니다. ID가 기록되지 않았다면 Session을 다시 조회해 확인하고, 없으면 명확한 오류로 중단합니다. 응답 환경 ID가 요청한 ID와 같은지도 확인합니다.

이름·유형·설명만 표시하고, 이번에 제공한 Skill의 이름과 `inline` 유형을 확인합니다. 이 목록은 등록 확인이며 실제 사용이나 기본 이미지의 전체 Skill 목록을 뜻하지 않습니다. 설치된 절대 경로를 이 metadata에서 추측하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting registration.")
    environment_id = sessions[result.session_id].get("environment_id")
    if not environment_id:
        session = client.beta.agents.sessions.retrieve(result.session_id, timeout=15)
        environment_id = getattr(session.environment, "id", None)
        sessions[result.session_id]["environment_id"] = environment_id
    if not environment_id:
        raise RuntimeError("No environment ID was captured for this session")
    installed_environment = client.beta.agents.environments.retrieve(environment_id, timeout=15)
    if installed_environment.id != environment_id:
        raise RuntimeError("Retrieved environment differs from the session environment")
    print("Registered custom Skills:", [
        {"name": s.name, "type": s.type, "description": s.description}
        for s in installed_environment.skills
    ])
    if not any(s.name == skill["name"] and s.type == "inline" for s in installed_environment.skills):
        raise RuntimeError("The requested inline Skill is not registered")


## 5. 이번 Turn의 실제 실행 명령 확인하기

`items.list()`를 SDK 페이지 반복으로 읽고 이번 root Turn의 명령만 표시합니다. 스크립트 경로·입력/출력 인자·작업 디렉터리·상태·종료 코드와 출력을 읽어 보세요. 종료 코드가 미보고인 `None`은 0이 아닙니다. 스크립트를 읽는 명령과 Python으로 실행하는 명령도 구분합니다.

별도 검증기는 같은 Turn에서 완료·종료 코드 0인 명시적인 Python CLI 실행과 의도한 입력/출력 인자를 확인합니다. 설치 경로는 고정하지 않습니다. 명령이 복잡한 shell 래퍼나 Python inline 코드로 감싸졌다면 직접 CLI 실행 형태로 확인할 수 있는지 조사하세요. 이 예제의 검사 범위는 명시적 CLI이며 일반 shell 분석기나 완전한 감사 로그가 아닙니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting command history.")
    items = list(client.beta.agents.sessions.items.list(
        result.session_id, order="asc", limit=100, timeout=15,
    ))
    command_items = [item for item in items
                     if item.type == "command_execution" and item.turn_id == result.turn_id]
    print("Commands from this turn:")
    for item in command_items:
        print({"item_id": item.id, "turn_id": item.turn_id, "command": item.command,
               "cwd": item.cwd, "status": item.status, "exit_code": item.exit_code,
               "output": item.output})


## 6. 세 결과 파일 다운로드하기

`artifacts.for_result(result)`로 완료한 Session·Turn에 범위를 고정합니다. 정확한 경로를 다운로드하고 원격 artifact와 로컬 경로·크기를 표시합니다. SDK가 누락·중복 경로를 오류로 처리합니다.

세 파일을 모두 받은 뒤에만 완료로 표시합니다. 재실행은 세션 정리 전에 한정합니다. 오류 시 정리를 시도하므로 같은 세션에서 재시도를 보장하지 않으며 부분 파일을 성공으로 처리하거나 분석을 자동 재실행하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before downloading artifacts.")
    downloads_complete = False
    downloads = output_dir / "downloads" / result.turn_id
    downloads.mkdir(parents=True, exist_ok=True)
    scoped = client.beta.agents.sessions.artifacts.for_result(result)
    for name in ("summary.json", "report.md", "skill_evidence.json"):
        local_path = downloads / name
        artifact = scoped.download(f"/workspace/outputs/{name}", to=local_path, timeout=30)
        print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
               "sandbox_path": artifact.path, "local_path": str(local_path),
               "local_bytes": local_path.stat().st_size})
    downloads_complete = True


## 7. 분석 결과와 실행 기록 읽기

| 항목 | 의미 |
| --- | --- |
| 로컬 ZIP SHA-256 | 요청에 담은 번들 바이트 식별 |
| `input_sha256` | 스크립트가 읽은 입력 바이트 식별 |
| `script_revision` | 코드 안에 정의한 revision 표식 |
| 명령 상태·종료 코드 | 실행 작업의 결과 |
| JSON·보고서 | 실제 생성한 산출물 |

최종 응답·로컬 `summary.json`·`skill_evidence.json`과 한국어 보고서를 읽습니다. 아래 정상 경로는 새 API 요청 없이 로컬 결과를 표시합니다. 해시·Skill 이름·revision·계산값·보고서의 기준값 대조는 별도 검증기가 수행합니다.

증거 JSON과 revision 문자열은 독립적인 서명이나 변조 방지를 보장하지 않습니다. 등록·명령·입력 해시·산출물을 함께 확인하는 실습 자료입니다. 기본 Skill의 언급이나 등록만으로 실제 사용을 확정하지 않습니다.


In [ ]:
from IPython.display import Markdown, display

with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not downloads_complete:
        raise RuntimeError("Complete all three downloads before reading local results.")
    actual = json.loads(result.output_text)
    downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
    skill_evidence = json.loads((downloads / "skill_evidence.json").read_text(encoding="utf-8"))
    report_text = (downloads / "report.md").read_text(encoding="utf-8")
    print("Final response:")
    print(json.dumps(actual, indent=2))
    print("Downloaded summary.json:")
    print(json.dumps(downloaded_summary, indent=2))
    print("Downloaded skill_evidence.json:")
    print(json.dumps(skill_evidence, indent=2))
    display(Markdown(report_text))


## 8. 세션 정리와 로컬 결과 보관

중간에 학습을 멈추더라도 정리 셀을 실행하세요. 필요한 파일은 세션 삭제 전에 내려받습니다. helper는 이번 실행의 metadata로 소유 세션을 찾아 필요하면 취소한 뒤 삭제·재조회 `404`로 API상 부재를 확인합니다.

출력의 `deleted`·`verified_absent`를 확인합니다. 물리적인 sandbox 정리의 동기 완료를 뜻하지 않습니다. 기존 실행·검증 오류가 있으면 정리 오류가 원래 원인을 가리지 않도록 처리합니다. 검증기는 삭제 전에 이력·파일 검사를 추가하며 실패해도 정리를 시도합니다.

로컬 Skill 원본·ZIP 바이트·다운로드·결과 객체는 남습니다. inline ZIP 실습은 별도 Skills API 업로드 개체나 저장 Agent를 만들지 않으므로 추가 삭제하지 않습니다. 일반 Jupyter 실행에는 정답 검사나 receipt를 추가하지 않습니다.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 9. 실패 시 조치와 재사용 범위

| 단계 | 확인할 내용 |
| --- | --- |
| 번들 준비 | manifest metadata·파일 누락·ZIP의 최상위 폴더 |
| 환경 준비·등록 | 첨부한 Skill·이번 Environment ID·이름/유형 |
| 스크립트 실행 | 실제 명령·CLI 인자·완료 상태·종료 코드 |
| 결과 수집 | 세 파일의 게시·다운로드·부분 전송·정리 결과 |

실패 명령이나 누락 증거를 정상 사용으로 판단하지 말고 8절 정리 후 원인을 확인하세요. 같은 요청을 무조건 재전송하지 않습니다. 키나 원본 API 오류 body는 출력하지 않습니다. 강제 종료 복구에는 `outputs/extensions/<chapter>/<run_id>/creation.json`과 Session ID를 사용합니다. 검증 도구의 실행 사본은 `outputs/extensions-validation/`에 있으며 결과·정리 기록은 `outputs/extensions/`에 있습니다. 복구 절차는 [빠른 시작](../quickstart.ipynb)을 참고하세요.

이 장은 inline ZIP 하나의 등록과 명시적 실행을 다룹니다. Skills API 업로드·버전 참조, Responses shell attachment, capability directory 사용은 요청 표면이 다릅니다. 다른 방식을 지원하지 않는다는 뜻이 아니라 이번 실습에서 함께 구현하지 않는다는 범위입니다.


## 완료 기준과 다음 장

Skill 파일의 역할, ZIP과 API 필드, 등록과 실행 증거, 해시·revision·산출물의 의미를 구분해 설명할 수 있어야 합니다. 실제 사용 확인은 자동 선택 품질이나 완전한 기업 감사 검증과 구분합니다.

[10장 Plugins](10_plugins.ipynb)에서는 이 Skill과 MCP 설정을 하나의 Plugin으로 묶습니다. [15장 Multi-agent](15_multi_agent.ipynb)와 Skill의 조합은 별도 선택 확장이며 현재 15장은 Skill을 첨부하지 않습니다.


### 등록 목록과 사용 증거

`environments.retrieve(...).skills`로 등록된 Custom Skill을 확인합니다. 이미지의 기본 스킬이 여기에 모두 표시되는 것은 아닙니다. 실제 사용은 다운로드한 증거 파일과 번들 스크립트 실행 이력을 검증 도구에서 대조합니다. [기본 스킬 확인 방법](../../docs/extensions/builtin-skills.md).


## 공식 참고 자료

### Agents API 전용 문서

- [OpenAI-hosted — Add skills and plugins](https://developers.openai.com/api/docs/guides/agents-api/environments/openai-hosted#add-skills-and-plugins) — hosted 환경의 skills·plugins·capability_directories 설정.

### 보조 자료 — 공통 문서

- [Skills — Agents API 절](https://developers.openai.com/api/docs/guides/tools-skills#agents-api) — 이 페이지는 Responses API도 함께 설명합니다. Agents API 절의 SKILL.md 탐색·capability_directories 설명을 읽으세요. 다른 절의 Skills API 업로드·버전 참조·shell attachment를 이 장의 inline ZIP 요청과 혼용하지 않습니다.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
